# Avance 4. Modelos alternativos

## Proyecto Integrador
### ThreatLLM: Clasificación de Ciberataques mediante Modelos de Lenguaje

**Equipo 55**

Integrantes

* Ada Jimena Vargas Aguirre — A01701619
* Edson Garduño Nolasco — A01795770
* Irving Alan García Zapata — A01796793


**Asesor**: Dr. Víctor Munguía




---

## Objetivos

3.3 Explorar una gama diversa de técnicas y enfoques con el fin de identificar el de mejor desempeño en el conjunto de datos en cuestión.

3.4 Encontrar la configuración óptima que maximiza el rendimiento del modelo en una tarea específica.

## Declaración de uso de IA y contribución del equipo

### Uso de herramientas de IA

| Herramienta utilizada | Propósito de uso | Alcance del apoyo (celdas o secciones) | Responsabilidad: qué revisó o validó el equipo |
|---|---|---|---|
| Anthropic PBC. (2026). Claude (Sonnet 5.5) [Modelo de lenguaje grande].<br> https://claude.ai| Verificación y optimización de código. Generación de fragmentos de código. |Celda 11, 12, 13,15 y 16 | Validar resultados entre iteraciones (que se respetara el uso de semilla) Cotejar resultados del baseline con el avance 3. |


### Contribución individual

| Integrante | Contribución al avance |
|---|---|
| Ada | Parte A |
| Edson | Parte B |
| Irving | Parte C |

### Decisiones principales

<small> A continuación se documentan las tres decisiones que más influyeron en el avance, indicando qué alternativa se descartó.</small>

<small><b>Decisión 1:</b> TF-IDF de caracteres sobre embeddings MiniLM.</small>

- <small>Justificación:
TF-IDF es mostró mejor rendimiento en LogReg (0.840 contra 0.802), Nearest Centroid (0.795 contra 0.746) y Ridge (0.837 contra 0.815).</small>
- <small>Alternativa descartada y motivo:
* MiniLM no ganó en ningún algoritmo (empata en SVM y kNN, pierde en LogReg, Nearest Centroid y Ridge).

* Gemma y mpnet quedaron fuera por tiempo.</small>

<small><b>Decisión 2:</b> Afinar LogReg y Complement NB</small>

- <small>Justificación: La comparación permitió evaluar las ventajas y limitaciones de cada alternativa con criterios comunes (F1, F2, recall, entre otros).

LogReg es la referencia del Av3 y es interpretable.

Complement NB: tiene el mejor recall (0.932) y F2 (0.878) antes de afinar.

</small>
- <small>Alternativa descartada y motivo: Se descartó SVM RBF, ya que empató con el baseline en F1 (0.841 contra 0.840) pero entrena en ~6 a 7 s contra ~1 s, sin ventaja clara.</small>

<small><b>Decisión 3:</b> Elegir Complement NB sobre LogReg por recall y F2.</small>

- <small>Justificación: En test el recall es 0.961 contra 0.789 del baseline, y los falsos negativos bajan de 70 a 13.</small>
- <small>Alternativa descartada y motivo: LogReg, que tiene mejor precisión (0.84 contra 0.69) e interpretabilidad.
IMPORTANTE: Esta elección es condicional a la respuesta del cliente.</small>


---

# Parte A | Exploración de modelos alternativos

En esta sección primero se convierte cada oración en un vector o *embedding* usando MiniLM en primera instancia, después se guardan en formato `.npy`; es decir, se calcula solo una vez para su uso posterior.

Acto seguido, se define la función `evaluar()`, con la cual se mide el desempeño de cada uno de los modelos probados. Para ello, se emplea validación cruzada de 5 pliegues en el conjunto train, se obtienen las métricas para el conjunto dev, así como el registro de tiempo de ejecución de cada modelo.

Luego se prueban **6 algoritmos distintos** (individuales, sin ensambles). En este caso se analizaron:
1. LogReg
2. Complement NB
3. SVM RBF
4. kNN
5. Árbol de decisión
6. MLP

Cada uno con dos representaciones cuando es posible:
* TF-IDF de caracteres
* Embeddings (NB solo con TF-IDF; MLP solo con embeddings).



NOTA:

**Diferencia con el equipo anterior (Equipo 11)**

Usaron 8 algoritmos, una sola representación TF-IDF de palabras, evaluados en dev.

Aquí hay validación cruzada con desviación, dos representaciones, tiempos de entrenamiento y predicción, y después ajuste fino.


## 0. Configurar entorno

In [1]:
#!pip -q install sentence-transformers          # Colab: quitamos el # la primera vez que se ejecutó
from pathlib import Path
import numpy as np, pandas as pd, time
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score,fbeta_score, make_scorer
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

In [2]:
# Definir rutas
DATA_DIR  = Path("data/ladder/sentence_classification")      # carpeta con train.csv, dev.csv, test.csv (separados por TAB)
KASPERSKY = Path("oraciones_kaspersky.csv")                  # id, tipo, texto (las 96 oraciones tienen tipo == "oracion")
EMB_NAME  = "minilm"                                         # "minilm" ; "mpnet"
OUT_DIR   = Path("."); SEED = 42

## 1. Cargar datos


In [3]:
# Descomprimir carpeta ZIP con archivos necesarios para la ejecución del notebook
# La carpeta debe cargarse manualmente al entorno de Colab.
!unzip -q -o data.zip
!ls
!ls data

data		    emb_minilm_kaspersky.npy  oraciones_kaspersky.csv
data.zip	    emb_minilm_test.npy       sample_data
emb_minilm_dev.npy  emb_minilm_train.npy
attack_pattern	ladder	mitre  reports


In [4]:
# Definir rutas/paths
tr = pd.read_csv(DATA_DIR / "train.csv", sep="\t"); dv = pd.read_csv(DATA_DIR / "dev.csv", sep="\t")
te = pd.read_csv(DATA_DIR / "test.csv",  sep="\t")
kas = pd.read_csv(KASPERSKY); kas = kas[kas["tipo"] == "oracion"].reset_index(drop=True)
print("train", tr.shape, "dev", dv.shape, "test", te.shape, "| oraciones reporte:", len(kas))
print("balance train:", tr.label.value_counts().to_dict())
ytr, ydv = tr.label.to_numpy(), dv.label.to_numpy()

train (2214, 2) dev (568, 2) test (662, 2) | oraciones reporte: 96
balance train: {1: 1107, 0: 1107}


## 2. Cálculo de embeddings

Primeramente, se definen las funciones para calcular y/o cargar los embeddings, según sea el caso.

Con esto presente, esta celda solo se ejecutó una vez.
Para ejecutar las celdas siguientes, se usaron los archivos `.npy` anexos.

Como primer experimento, se utilizó el modelo `all-MiniLM-L6-v2` por ser un codificador de oraciones y parráfos cortos, de peso ligero, propósito general y gratuito.

In [5]:
# Función para crear los embeddings
def embeber(textos, nombre_modelo):
  from sentence_transformers import SentenceTransformer
  ids = {"minilm": "sentence-transformers/all-MiniLM-L6-v2", "mpnet": "sentence-transformers/all-mpnet-base-v2"}
  modelo = SentenceTransformer(ids[nombre_modelo])
  return modelo.encode(list(textos),batch_size = 64,normalize_embeddings = True, show_progress_bar=True)


# Función para cargar embeddings almacenados en .npy
def embeddings_cacheados(nombre):
  out, partes = {}, {"train": tr.text, "dev": dv.text, "test": te.text, "kaspersky": kas["texto"]}

  for k, t in partes.items():
        f = OUT_DIR / f"emb_{nombre}_{k}.npy"
        # Se carga si ya existe el archivo
        if f.exists():
          out[k] = np.load(f)
        # En caso de que no se tengan embeddings cargados, se crean y guardan
        else:
            t0 = time.time()
            out[k] = embeber(t, nombre)
            np.save(f, out[k])
            print(f"  {k}: {out[k].shape} guardado en {f.name} ({time.time()-t0:.0f}s)")
  return out

E = embeddings_cacheados(EMB_NAME)
print({k: v.shape for k, v in E.items()})


{'train': (2214, 384), 'dev': (568, 384), 'test': (662, 384), 'kaspersky': (96, 384)}


In [6]:
# Comprobar normalización (escala 0-1)
np.linalg.norm(E["train"], axis=1)[:5]

array([0.99999994, 1.        , 1.        , 0.99999994, 1.        ],
      dtype=float32)

## 3. Evaluación inicial de modelos alternativos

Se definen las funciones necesarias para la medición de desempeño de los modelos y presentación de resultados.

In [7]:
RESULTADOS = []

# Función para evaluar los modelos seleccionados
def evaluar(nombre, modelo, Xtr, ytr, Xdev, ydev, rep="", notas=""):
    """Misma medición para todos:
    - validación cruzada de 5 pliegues SOLO en train (métricas: F1, recall, precisión; media ± desviación)
    - entrenar en todo train (se mide el tiempo) y evaluar en dev (se mide el tiempo de predicción)
    `modelo` puede ser un Pipeline (TF-IDF + clasificador: el vectorizador se ajusta dentro de cada pliegue) o un clasificador
    sobre embeddings. Se guarda una fila en la lista RESULTADOS."""

    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    scoring = {"f1":"f1", "recall":"recall", "precision":"precision",
               "f2":make_scorer(fbeta_score, beta =2)}
    r = cross_validate(modelo, Xtr, ytr, cv=cv, scoring=scoring, n_jobs=1)
    t0 = time.perf_counter(); modelo.fit(Xtr, ytr); t_fit = time.perf_counter() - t0
    t0 = time.perf_counter(); p = modelo.predict(Xdev); t_pred = time.perf_counter() - t0

    fila = {"modelo": nombre, "representación": rep,
            "cv_f1": round(r["test_f1"].mean(), 3), "cv_f1_std": round(r["test_f1"].std(), 3),
            "cv_recall": round(r["test_recall"].mean(), 3), "cv_precision": round(r["test_precision"].mean(), 3),
            "cv_f2": round(r["test_f2"].mean(), 3), "cv_f2_std": round(r["test_f2"].std(), 3),
            "dev_acc": round(accuracy_score(ydev, p), 3), "dev_f1": round(f1_score(ydev, p), 3),
            "dev_recall": round(recall_score(ydev, p), 3), "dev_precision": round(precision_score(ydev, p), 3),
            "dev_f2": round(fbeta_score(ydev, p, beta=2), 3),
            "t_entrena_s": round(t_fit, 2), "t_predice_dev_s": round(t_pred, 3), "notas": notas}

    RESULTADOS.append(fila); return fila

# Función para convertir lista a dataframe y ordenar los valores
def tabla():
    return pd.DataFrame(RESULTADOS).sort_values("cv_f1", ascending=False).reset_index(drop=True)

Se agrega el baseline del avance previo para fines de comparación.

In [8]:
# Baseline del Avance 3, NO cuenta entre los 6 modelos
baseline = make_pipeline(TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), sublinear_tf=True, min_df=1),
                         LogisticRegression(C=3, class_weight="balanced", solver="liblinear", max_iter=5000, random_state=SEED))

evaluar("REFERENCIA Av3: LogReg", baseline, tr.text, ytr, dv.text, ydv, rep="TF-IDF char (3,6)", notas="baseline Av3")

{'modelo': 'REFERENCIA Av3: LogReg',
 'representación': 'TF-IDF char (3,6)',
 'cv_f1': np.float64(0.84),
 'cv_f1_std': np.float64(0.009),
 'cv_recall': np.float64(0.844),
 'cv_precision': np.float64(0.837),
 'cv_f2': np.float64(0.842),
 'cv_f2_std': np.float64(0.017),
 'dev_acc': 0.815,
 'dev_f1': 0.821,
 'dev_recall': 0.849,
 'dev_precision': 0.795,
 'dev_f2': 0.837,
 't_entrena_s': 0.64,
 't_predice_dev_s': 0.126,
 'notas': 'baseline Av3'}

In [9]:
#  8 ALGORITMOS diferentes (cada uno con 1 o 2 representaciones según sea el caso)
# Se incluye LogReg para referencia y comparación según la representación empleada (embeddings vs TF-IDF)

from sklearn.linear_model import RidgeClassifier
from sklearn.neighbors import KNeighborsClassifier, NearestCentroid
from sklearn.naive_bayes import GaussianNB, ComplementNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC

def tfidf_char():   # TF-IDF de caracteres, igual que la baseline del Avance 3 (el vectorizador se ajusta dentro de cada pliegue)
    return TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 6), sublinear_tf=True, min_df=1)

Xtr, Xdv = E["train"], E["dev"]
T_TXT, D_TXT = tr.text, dv.text
TF = "TF-IDF char (3,6)"

# Algoritmo 0: Regresión logística (lineal)
evaluar("0 LogReg", LogisticRegression(C=1, max_iter=2000, random_state=SEED), Xtr, ytr, Xdv, ydv, rep=EMB_NAME)
# (la fila de TF-IDF de este algoritmo es la REFERENCIA del Av3, ya medida arriba)

# Algoritmo 1: Gaussian Naive Bayes
evaluar("1 Gaussian NB", GaussianNB(), Xtr, ytr, Xdv, ydv,rep = EMB_NAME)

# Algoritmo 2: Complement Naive Bayes (probabilístico). Solo TF-IDF porque NB necesita valores no negativos
evaluar("2 Complement NB", make_pipeline(tfidf_char(), ComplementNB(alpha=0.5)), T_TXT, ytr, D_TXT, ydv, rep=TF,
        notas="alpha provisional")

# Algoritmo 3: SVM con kernel RBF (frontera no lineal)
evaluar("3 SVM RBF", SVC(C=1, kernel="rbf", gamma="scale", random_state=SEED), Xtr, ytr, Xdv, ydv, rep=EMB_NAME)
evaluar("3 SVM RBF", make_pipeline(tfidf_char(), SVC(C=1, kernel="rbf", gamma="scale", random_state=SEED)),
        T_TXT, ytr, D_TXT, ydv, rep=TF)

# Algoritmo 4: k vecinos más cercanos (basado en instancias)
evaluar("4 kNN (coseno)", KNeighborsClassifier(n_neighbors=15, metric="cosine", algorithm="brute"), Xtr, ytr, Xdv, ydv,
        rep=EMB_NAME, notas="k=15 provisional")
evaluar("4 kNN (coseno)", make_pipeline(tfidf_char(), KNeighborsClassifier(n_neighbors=15, metric="cosine", algorithm="brute")),
        T_TXT, ytr, D_TXT, ydv, rep=TF, notas="k=15 provisional")

# Algoritmo 5: Nearest Centroid/ Rocchio Classifier, se usa la métrica default (”euclidean”)
evaluar("5 Nearest Centroid",NearestCentroid(),Xtr,ytr,Xdv,ydv, rep = EMB_NAME)
evaluar("5 Nearest Centroid",make_pipeline(tfidf_char(),NearestCentroid()),T_TXT,ytr,D_TXT,ydv, rep = TF)

# Algoritmo 6: Ridge Classifier
evaluar("6 Ridge Classifier",RidgeClassifier(),Xtr,ytr,Xdv,ydv, rep = EMB_NAME)
evaluar("6 Ridge Classifier",make_pipeline(tfidf_char(),RidgeClassifier()),T_TXT,ytr,D_TXT,ydv, rep = TF)

# Algoritmo 7: Árbol de decisión (un solo árbol, NO es bosque)
evaluar("7 Árbol de decisión", DecisionTreeClassifier(max_depth=12, random_state=SEED), Xtr, ytr, Xdv, ydv, rep=EMB_NAME,
        notas="max_depth provisional")
evaluar("7 Árbol de decisión", make_pipeline(tfidf_char(), DecisionTreeClassifier(max_depth=12, random_state=SEED)),
        T_TXT, ytr, D_TXT, ydv, rep=TF, notas="max_depth provisional; puede tardar más")

# Algoritmo 8: Red neuronal MLP (una red pequeña; NO es ensamble). Solo embeddings (con 63k rasgos sería lenta) ---
evaluar("8 MLP", MLPClassifier(hidden_layer_sizes=(128,), alpha=1e-3, early_stopping=True, max_iter=300, random_state=SEED),
        Xtr, ytr, Xdv, ydv, rep=EMB_NAME, notas="128 neuronas provisional")



{'modelo': '8 MLP',
 'representación': 'minilm',
 'cv_f1': np.float64(0.802),
 'cv_f1_std': np.float64(0.011),
 'cv_recall': np.float64(0.848),
 'cv_precision': np.float64(0.761),
 'cv_f2': np.float64(0.829),
 'cv_f2_std': np.float64(0.015),
 'dev_acc': 0.761,
 'dev_f1': 0.755,
 'dev_recall': 0.739,
 'dev_precision': 0.772,
 'dev_f2': 0.746,
 't_entrena_s': 1.7,
 't_predice_dev_s': 0.002,
 'notas': '128 neuronas provisional'}

In [10]:
T = tabla(); print(T.to_string(index=False))
T.to_csv(OUT_DIR / f"resultados_av4_bloque1_{EMB_NAME}.csv", index=False)

                modelo    representación  cv_f1  cv_f1_std  cv_recall  cv_precision  cv_f2  cv_f2_std  dev_acc  dev_f1  dev_recall  dev_precision  dev_f2  t_entrena_s  t_predice_dev_s                                   notas
             3 SVM RBF TF-IDF char (3,6)  0.841      0.010      0.836         0.847  0.838      0.018    0.813   0.817       0.835          0.801   0.828         7.06            1.425                                        
REFERENCIA Av3: LogReg TF-IDF char (3,6)  0.840      0.009      0.844         0.837  0.842      0.017    0.815   0.821       0.849          0.795   0.837         0.64            0.126                            baseline Av3
             3 SVM RBF            minilm  0.838      0.007      0.858         0.820  0.850      0.011    0.787   0.792       0.810          0.774   0.803         0.29            0.177                                        
    6 Ridge Classifier TF-IDF char (3,6)  0.837      0.013      0.839         0.836  0.838      0.023   

### Hallazgos de la comparativa

A modo de contexto, vale la pena señalar que la tabla anterior está ordenada por `cv_f1` (la cual fue seleccionada como métrica principal en la entrega previa), y que la desviación (`cv_f1_std`) permite identificar si una diferencia es ruido; es decir, si la diferencia observada entre las puntuaciones F1 de los modelos es pequeña respecto a la variabilidad de los resultados, no se tiene suficiente evidencia para asegurar que un modelo tenga un desempeño superior.

Con esto en mente, se describen los hallazgos:
1. **F1**

Los modelos que quedan prácticamente empatados en F1 son SVM RBF y LogReg, ambos con la representación de TF-IDF, porque sus medias (`cv_f1` = 0.841 y 0.840, respectivamente) difieren menos que su desviación estándar.

Asimismo, aunque no es concluyente, LogReg presenta un valor mayor en su desempeño en F1 sobre el conjunto dev (0.821 contra 0.817).

Por otro lado, existen otros modelos que también quedan empatados: SVM usando embeddings (0.838 ± 0.007) y Ridge con TF-IDF (0.837 ± 0.013). Sin embargo, ambos tienen un desempeño inferior que la primera pareja al ser evaluados sobre el conjunto dev (F1 de 0.792 y 0.807, respectivamente).

---

2. **¿Qué implica el empate para elegir?**

Dado que no es posible seleccionar un claro ganador por la métrica F1, este empate implica considerar otros valores obtenidos como criterios de selección: las métricas recall y F2, así como el tiempo de entrenamiento.


---
3. **Recall**

Al revisar el recall, los dos modelos que más lo favorecen son Complement NB con TF-IDF (`cv_recall` = 0.932) y kNN con embeddings (`cv_recall`= 0.921).

Esto a costa de su precisión tanto en validación cruzada como en su evaluación sobre dev.

Con un modelo así, puede significar que habría mayor cantidad de falsos positivos. En este caso, se tendría que sopesar si uno está dispuesto a tener más falsas alarmas, pero teniendo la mayor cantidad de verdaderos positivos.

---

4. **F2**

Ahora bien, al considerar F2, el orden cambia respecto a F1 y, nuevamente, Complement NB con TF-IDF (0.878 ± 0.016) y kNN con embeddings (0.866 ± 0.011) muestran los mejores desempeños en validación cruzada.

La diferencia entre ambos modelos es de 0.012, la cual es menor que la desviación (0.016), entonces no es posible decir que uno sea mejor que el otro.

Ahora, si se compara Complement NB (0.878) contra el baseline (0.842 ± 0.017), la diferencia es 0.036. Esta es aproximadamente el doble de la desviación (0.017) por lo que, probablemente, la mejora es consistente, no solo azar de la partición.

---
5. **Embeddings frente a TF-IDF**

Al comparar la misma técnica con ambas representaciones, se aprecia que en ningún algoritmo los embeddings superan a TF-IDF, sin embargo, en algunos empatan.

En los algoritmos LogReg (0.038), Nearest Centroid (0.049) y Ridge (0.022), la diferencia es mayor que las desviaciones.

En el caso de SVM RBF (0.003) y kNN (0.005) las diferencias son menores a la desviación así que están en empate.

En el caso del árbol de decisión (0.032), este queda en el límite, con desviaciones de 0.022 y 0.024.

También es importante mencionar que hay modelos que solo usaron una representación por la forma en que estos funcionan:
* Gaussian NB (embeddings) -> requiere matrices densas, TF-IDF es dispersa
* Complement NB (TF-IDF) -> necesita valores no negativos y los embeddings tienen negativos
* MLP (embeddings) -> se optó por embeddings porque usar 63,000 rasgos (TF-IDF) haría que fuera lento.

Por otro lado, a primer vistazo, los tiempos de entrenamiento disminuyen al usar los embeddings. No obstante, no son del todo comparables porque el tiempo para TF-IDF incluye construir el vectorizador dentro del pipeline, y el de embeddings no incluye calcularlos (se hizo una sola vez).

Es importante destacar que en kNN el tiempo de entrenamiento no es informativo, pues el modelo solo almacena datos. Su costo como tal está en el tiempo de predicción, y en la versión TF-IDF el tiempo de entrenamiento incluye el cálculo del vectorizador.


Con lo anterior presente, para el ajuste fino se eligen como dos modelos: LogReg + TF-IDF, y Complement NB + TF-IDF.

El primero por su desempeño en F1, su tiempo de entrenamiento (0.56 s contra 6.14 s de SVM TF-IDF), la interpretabilidad de los coeficientes y que su C = 3 es un valor heredado sin ajuste.

El segundo modelo se selecciona por su rendimiento ligeramente mayor en F2 y recall, en comparación con kNN. Además, su hiperparámetro principal (`alpha`) es sencillo de explorar. Se reconoce el compromiso que implica en F1 y precisión.

Los hiperparámetros (C y tipo de penalización en LogReg; alpha en Complement NB; y los parámetros de TF-IDF) no están afinados, así que se procede a realizar ajuste fino.

## 4. Ajuste fino (Fine Tuning)

Primero se define la configuración de la búsqueda de hiperparámetros.

In [11]:
import time
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, cross_validate
from scipy.stats import loguniform
from sklearn.metrics import make_scorer, fbeta_score

# Las mismas 5 pruebas (misma semilla = mismos cortes del train)
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)

# Las 4 métricas que se miden en cada intento
# F2 se arma con beta = 2, así pesa más el recall
scoring = {"f1": "f1", "recall": "recall", "precision": "precision",
           "f2": make_scorer(fbeta_score, beta=2)}

# Función para probar n_iter combinaciones al azar.
# Se queda con la mejar según 'metrica'
def buscar(pipe, espacio, n_iter=40, metrica = "f1"):
    t0 = time.perf_counter()  # cronómetro
    bs = RandomizedSearchCV(pipe, espacio, n_iter=n_iter, scoring=scoring,
                            refit=metrica, # la metrica con la que se elige al "ganador"
                            cv=cv, n_jobs=-1, random_state=SEED)

    bs.fit(tr.text, ytr)      # SOLO train
    i = bs.best_index_        # posición del intento ganador

    print(f"mejor cv_{metrica} = {bs.best_score_:.3f} ± {bs.cv_results_[f'std_test_{metrica}'][i]:.3f}"
          f" | {time.perf_counter()-t0:.0f} s | {bs.best_params_}")
    return bs

# Ajustes para el TF-IDF (es igual para ambos modelos)
tfidf_esp = {"tfidfvectorizer__ngram_range": [(2, 5), (3, 6), (3, 7), (4, 6)],  #largo de los n-gramas
             "tfidfvectorizer__min_df": [1, 2, 3],                              # mínimo de oraciones donde debe aparecer un n-grama
             "tfidfvectorizer__sublinear_tf": [True, False]}                    # suavizar repeticiones

# LogReg
# C = qué tan "suelto" es el modelo o qué tanta libertad se le da al modelo para ajustarse al train
# penalty = tipo de penalización
pipe_lr = make_pipeline(TfidfVectorizer(analyzer="char_wb"),
                        LogisticRegression(class_weight="balanced", solver="liblinear",
                                           max_iter=5000, random_state=SEED))
esp_lr = {**tfidf_esp, "logisticregression__C": loguniform(1e-2, 1e2),
          "logisticregression__penalty": ["l1", "l2"]}

# Complement NB
# alpha = suavizado
# norm = normalizar los pesos
pipe_nb = make_pipeline(TfidfVectorizer(analyzer="char_wb"), ComplementNB())
esp_nb = {**tfidf_esp, "complementnb__alpha": loguniform(1e-3, 10),
          "complementnb__norm": [True, False]}

# Correr la búsqueda (60 intentos cada una)
bs_lr = buscar(pipe_lr, esp_lr, n_iter = 60, metrica = "f1") # LogReg se elige por F1 (lo equilibrado)
bs_nb = buscar(pipe_nb, esp_nb, n_iter = 60, metrica = "f2") # CNB se elige por F2 (porque importa el recall)

mejor cv_f1 = 0.841 ± 0.006 | 153 s | {'logisticregression__C': np.float64(3.382387201936218), 'logisticregression__penalty': 'l2', 'tfidfvectorizer__min_df': 2, 'tfidfvectorizer__ngram_range': (3, 7), 'tfidfvectorizer__sublinear_tf': True}
mejor cv_f2 = 0.890 ± 0.009 | 156 s | {'complementnb__alpha': np.float64(0.11040511903162252), 'complementnb__norm': True, 'tfidfvectorizer__min_df': 2, 'tfidfvectorizer__ngram_range': (2, 5), 'tfidfvectorizer__sublinear_tf': True}


Ahora, para tener forma de comparar se define una función para construir la tabla de comparación: antes del ajuste fino vs después con ajuste aplicado.

In [12]:
# Definir función para tabla comparativa
# Se compara antes vs después (en validación cruzada, solo train)

def medir(nombre, modelo):   # Es el ANTES
    # Corre la misma validación cruzada de siempre (mismos pliegues, mismas métricas)
    # Resume en 'promedio +- desviación'
    r = cross_validate(modelo, tr.text, ytr, cv=cv, scoring=scoring, n_jobs=-1)
    fila = {"modelo": nombre}
    for m in ["f1", "recall", "f2"]:
        # r["test_f1"] trae 5 números (uno por pliegue): sacamos promedio ± desviación
        fila[f"cv_{m}"] = f"{r['test_'+m].mean():.3f} ± {r['test_'+m].std():.3f}"
    return fila

def mejor_de(nombre, bs):   # Es el DESPUÉS
    # cv_results_ guarda los resultados de TODOS los intentos
    # best_index_ es la posición del ganador
    r = pd.DataFrame(bs.cv_results_).iloc[bs.best_index_]
    fila = {"modelo": nombre}
    for m in ["f1", "recall", "f2"]:
        fila[f"cv_{m}"] = f"{r['mean_test_'+m]:.3f} ± {r['std_test_'+m]:.3f}"
    return fila

# Construir e imprimir tabla comparativa
antes_despues = pd.DataFrame([
    medir("LogReg ANTES (baseline, C=3)", baseline),                                   # el baseline del Av3
    mejor_de("LogReg DESPUÉS", bs_lr),
    medir("Complement NB ANTES (alpha=0.5)", make_pipeline(tfidf_char(), ComplementNB(alpha=0.5))),
    mejor_de("Complement NB DESPUÉS", bs_nb)])
print(antes_despues.to_string(index=False))


                         modelo         cv_f1     cv_recall         cv_f2
   LogReg ANTES (baseline, C=3) 0.840 ± 0.009 0.844 ± 0.024 0.842 ± 0.017
                 LogReg DESPUÉS 0.841 ± 0.006 0.842 ± 0.022 0.841 ± 0.014
Complement NB ANTES (alpha=0.5) 0.809 ± 0.016 0.932 ± 0.017 0.878 ± 0.016
          Complement NB DESPUÉS 0.804 ± 0.011 0.958 ± 0.011 0.890 ± 0.009


In [13]:
# Revisión en dev (una vez por modelo)
_ = evaluar("LogReg afinado", bs_lr.best_estimator_, tr.text, ytr, dv.text, ydv, rep=TF)
_ = evaluar("Complement NB afinado", bs_nb.best_estimator_, tr.text, ytr, dv.text, ydv, rep=TF)


In [14]:
T = tabla()     # tabla completa de todos los modelos evaluados
T[T["modelo"].str.contains("afinado")][["modelo", "dev_f1", "dev_recall", "dev_precision", "dev_f2"]]

,modelo,dev_f1,dev_recall,dev_precision,dev_f2
1,LogReg afinado,0.816,0.838,0.796,0.829
7,Complement NB afinado,0.772,0.968,0.643,0.879


In [15]:
# Tiempos
# Estos cambian un poco entre ejecuciones del cuaderno

for nombre, modelo in [("LogReg afinado", bs_lr.best_estimator_),
                       ("Complement NB afinado", bs_nb.best_estimator_)]:
    t0 = time.perf_counter()
    modelo.fit(tr.text, ytr)    # entrenar con train
    t_fit = time.perf_counter() - t0

    t0 = time.perf_counter()
    modelo.predict(dv.text)     # predecir dev
    t_pred = time.perf_counter() - t0

    print(f"{nombre}: entrenamiento {t_fit:.2f} s | predicción en dev {t_pred:.3f} s")

LogReg afinado: entrenamiento 1.01 s | predicción en dev 0.191 s
Complement NB afinado: entrenamiento 0.75 s | predicción en dev 0.160 s


In [16]:
# Diagnósticos de la búsqueda

# a. Rango de resultados: ¿qué tanto cambia el score entre los 60 intentos?
for bs, m in [(bs_lr, "f1"), (bs_nb, "f2")]:
    c = pd.DataFrame(bs.cv_results_)[f"mean_test_{m}"]
    print(f"{m}: mínimo {c.min():.3f}, máximo {c.max():.3f}")

# b. ¿En qué intento aparació el mejor? ¿dónde se presenta la meseta?
for bs, m in [(bs_lr, "f1"), (bs_nb, "f2")]:
    c = pd.DataFrame(bs.cv_results_)[f"mean_test_{m}"]
    print(f"{m}: el mejor apareció en el intento {int(c.idxmax()) + 1} de {len(c)}"
          f" | intentos a menos de 0.005 del mejor: {(c >= c.max() - 0.005).sum()}")

# c. Para LogReg específicamente
# Se buscan los intentos con F1 = 0 (el modelo se "apagó" y predice solo una clase)
cr = pd.DataFrame(bs_lr.cv_results_)
print("intentos con F1 = 0:", (cr.mean_test_f1 == 0).sum())
cr[cr.mean_test_f1 == 0][["param_logisticregression__penalty", "param_logisticregression__C"]]


f1: mínimo 0.000, máximo 0.841
f2: mínimo 0.823, máximo 0.890
f1: el mejor apareció en el intento 51 de 60 | intentos a menos de 0.005 del mejor: 3
f2: el mejor apareció en el intento 41 de 60 | intentos a menos de 0.005 del mejor: 10
intentos con F1 = 0: 9


,param_logisticregression__penalty,param_logisticregression__C
18,l1,0.060804
24,l1,0.019778
31,l1,0.027013
37,l1,0.198326
38,l1,0.027555
49,l1,0.089238
51,l1,0.022970
52,l1,0.055728
59,l1,0.110488


### Hallazgos del ajuste fino


A continuación se describen los hallazgos reunidos para cada modelo.

**LogReg**

Se ajustaron `C`, la penalización y los parámetros de TF-IDF. En este caso, la mejor configuración encontrada fue:
  * `C` = 3.38 (heredado era 3)
  * penalización `l2`, la cual penaliza los pesos grandes (misma penalización del baseline).
  * `min_df` = 2, es la cantidad mínima de oraciones distintas en las que debe aparecer un "trozo" de letras para que el modelo lo considere.
  * n-gramas (3,7), es el tamaño de los trozos de letras con los que el modelo mira cada palabra.
  
Al comparar antes y después del ajuste fino (en validación cruzada) se observó que el cambio en `F1` fue pequeño (0.840 -> 0.841), y que el `recall` (0.844 -> 0.842) y `F2` (0.842 -> 0.841) prácticamente no cambiaron.

En otras palabras, todas las diferencias son aproximadamente 0.1 veces la desviación, lo que sugiere que no se puede determinar una mejora más allá del vaivén del propio modelo.

En dev (antes -> después):
  * `F1`: 0.821 -> 0.816
  * `recall`: 0.849 -> 0.838
  * Precisión: 0.795 -> 0.796
  * `F2`: 0.837 -> 0.829

En dev, no se observó una mejoría del modelo afinado frente al baseline.

Por otro lado, el valor de `C` es similar al heredado y está dentro del espacio de búsqueda explorado, lo que apunta a que el baseline estaba cerca de ser óptimo.

Además, en 9 de los 60 intentos se obtuvo `F1` = 0; todos usaron `l1` con `C` entre 0.02 y 0.20.

---
**Complement NB**

Los parámetros ajustados fueron:
* `alpha` = 0.110
* `norm` = True
* `min_df` = 2
* n-gramas (2,5)

Al comparar antes y después del ajuste fino, se notaron los siguientes cambios en las métricas durante la validación cruzada:

* `recall`: 0.932 -> 0.958
* `F2`: 0.878 -> 0.890
* `F1`: 0.809 -> 0.804
* Precisión: 0.714 -> 0.692

Mientras que en dev se tuvo:
  * `recall`: 0.940 -> 0.968
  * `F2`: 0.876 -> 0.879
  * `F1`: 0.795 -> 0.772
  * Precisión: 0.688 -> 0.643

Se observa que el `recall` alcanzó 1.5 veces su desviación, F2 se quedó cerca (0.8 veces).

Mientras que se ganó en recall y F2 (de forma marginal), se perdió precisión y `F1`.

El rango de `F2` en toda la búsqueda (los 60 intentos) fue de 0.823 a 0.890, lo que indica que el ajuste importa para este modelo (considerando que la diferencia entre el peor y mejor intento es de 0.067, y la desviación del modelo consigo mismo es de 0.009 a 0.016, por lo que la diferencia es al menos 4 veces la desviación).


## Modelo individual final

Se selecciona como modelo individual final a **Complement NB + TF-IDF**, esto porque se considera que las métricas con mayor peso para el negocio son `recall` y `F2`.

El costo (trade-off) está en la precisión (0.692 en CV, 0.643 en dev) y F1; donde aproximadamente 1 de cada 3 oraciones marcadas es falsa alarma. Aquí vale la pena recordar que un falso negativo (una amenaza real) se puede pasar por alto y no hay forma de recuperarla en etapas posteriores.

En cuanto a interpretabilidad, en LogReg cada trozo de letras (n-grama de caracteres) tiene un coeficiente con signo (positivo empuja hacia "ataque", negativo hacia "contexto"), lo cual es más directo al leer. En Complement NB cada trozo tiene un peso por clase, pero la lectura no es tan intuitiva porque los pesos son log-probabilidades. Para un analista de SOC, sería más fácil explicarle el modelo LogReg. Este es otro trade-off a tener en cuenta.

Respecto a tiempos, tanto LogReg (entrenamiento: 1.01 s, predicción en dev: 0.191 s) como Complement NB (entrenamiento: 0.75 s, predicción en dev: 0.160 s) son similares.

Se realiza esta selección para este avance, sin embargo está sujeta a la confirmación pendiente con el cliente; si este prefiere precisión sobre recall, LogReg es la alternativa.

In [17]:
yte = te.label.to_numpy()   # etiquetas de test, igual que ytr y ydv

In [18]:
# Evaluación final en test (una sola vez)

from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, fbeta_score, confusion_matrix

modelo_final = bs_nb.best_estimator_

pred = modelo_final.predict(te.text)

print(f"accuracy : {accuracy_score(yte, pred):.3f}")
print(f"F1       : {f1_score(yte, pred):.3f}")
print(f"recall   : {recall_score(yte, pred):.3f}")
print(f"precisión: {precision_score(yte, pred):.3f}")
print(f"F2       : {fbeta_score(yte, pred, beta=2):.3f}")

# Matriz de confusión
# filas = real (0 contexto, 1 ataque)
# columnas = predicho
print(confusion_matrix(yte, pred))

accuracy : 0.776
F1       : 0.811
recall   : 0.961
precisión: 0.702
F2       : 0.895
[[196 135]
 [ 13 318]]


### Hallazgo del modelo individual final sobre test

Se aprecia que el comportamiento del modelo descrito anteriormente en CV y dev se repite en test:
* `recall` alto (0.96)
* precisión baja (0.70)

| | Baseline Av3 (LogReg) | Complement NB afinado |
|---|---|---|
| Falsos negativos | 70 | **13** |
| Falsos positivos | 51 | 135 |
| Recall | 0.789 | **0.961** |
| F1 | 0.812 | 0.811 |
| Accuracy | 0.817 | 0.776 |

NOTA: valores sobre la misma partición de test, evaluada una sola vez.

Complement NB cumple los dos umbrales fijados en el Avance 3 (F1 ≥ 0.75 y recall ≥ 0.80), mientras que el baseline no cumple el de recall (0.789).

Si bien `accuracy` es de 0.776 (menor al baseline) y `F1` es de 0.811 (empatada al baseline), el modelo Complement NB gana en `recall`, pagando en precisión y `accuracy`. Esto es esperado, y se discutió anteriormente al darle prioridad a reducir falsos negativos.

`accuracy` disminuye porque el modelo reduce 57 falsos negativos (70 a 13), a costa de 84 falsos positivos adicionales. Además, esta métrica cuenta los dos errores como iguales, por ende, cualquier modelo que intercambie un tipo de error por otro, la hará bajar.

Con base en el contexto de negocio, el cambio permite compensar si un falso negativo cuesta más de ~1.5 falsos positivos (84 / 57 ≈ 1.47) y esto se somete a la confirmación del cliente.

Como hipótesis, se plantea que la etapa de similitud con MITRE podría filtrar las falsas alarmas. Por limitante de tiempo, esto queda como trabajo futuro.




# Parte B | Similitud

# Parte C | Análisis de sensibilidad para `Composite Threat Score`